[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/theochem/grid/blob/master/examples/Sobol.ipynb)

# Sobol' Sequences

The [Sobol](https://grid.qcdevs.org/pyapi/grid.sobol.html#grid.sobol.Sobol)
grid can be used for integration over a (hyper)cubic or parallelepiped domain.
Sobol' sequences (Sobol', 1967) are a digital-net construction: points are
generated in base 2 so that, for any dimension, the elementary base-2
intervals are filled as evenly as possible. Unlike Monte Carlo, whose error
decreases as $O(N^{-1/2})$, Sobol' sequences achieve $O((\log N)^d / N)$ for
sufficiently smooth (bounded-variation) integrands -- substantially faster
for large $N$. This notebook builds a Sobol' design, visualizes it in three
dimensions, and compares it against baselines already in `grid` on two
integrands from computational chemistry -- one smooth, one not -- to see
honestly where this advantage does and does not hold.

## Initialization of Sobol

`Sobol` is initialized by specifying:

1. `n_points` -- the number of integration points $N$. Must be a power of 2
   (e.g. 1024, 2048, ...), for the balance properties of the underlying
   digital-net construction.
2. `dimension` -- the dimension $d$ of the integration domain.
3. `seed` (optional) -- for reproducibility, used only when `randomize=True`.
4. `randomize` (optional, default `True`) -- if `True`, applies Owen
   scrambling to the sequence. If `False`, generates the deterministic,
   unscrambled Sobol' sequence (whose first point is always the origin).
5. `origin` and `axes` (optional) -- to map the design from the unit
   hypercube $[0,1)^d$ onto an arbitrary parallelepiped.


In [ ]:
from grid.sobol import Sobol
import numpy as np

sobol = Sobol(n_points=16, dimension=2, seed=0)
print(f"Number of points: {sobol.size}")
print(f"Dimension: {sobol.dimension}")
print(f"Points are in [0, 1)^2: {(sobol.points >= 0).all() and (sobol.points < 1).all()}")
print()
print("Points:")
print(sobol.points)

With `randomize=False`, the sequence is deterministic and its very first
point is always the origin -- a defining property of the unscrambled Sobol'
construction.

In [ ]:
sobol_plain = Sobol(n_points=16, dimension=2, seed=0, randomize=False)
sobol_scrambled = Sobol(n_points=16, dimension=2, seed=0, randomize=True)

print("First point, randomize=False:", sobol_plain.points[0])
print("First point, randomize=True: ", sobol_scrambled.points[0])

By default, points live on the unit hypercube $[0,1)^d$. Passing `origin`
and `axes` maps the design onto an arbitrary parallelepiped, following the
same convention as `Lattice` and `LatinHypercube`.

In [ ]:
origin = np.array([1.0, 2.0])
axes = np.array([[3.0, 0.0], [0.0, 0.5]])
sobol_mapped = Sobol(n_points=16, dimension=2, seed=0, origin=origin, axes=axes)

print(f"Points now lie in [1, 4) x [2, 2.5): "
      f"{(sobol_mapped.points[:, 0] >= 1).all() and (sobol_mapped.points[:, 0] < 4).all()}")
print(f"Weight per point (volume / N): {sobol_mapped.weights[0]:.4f}")

## A 3D Scatter Plot of the Design

Sobol' points fill space evenly across scales: doubling $N$ refines the
existing pattern rather than replacing it (the first $N$ points of a
$2N$-point design are exactly the same $N$ points), which is easy to state
but worth seeing directly.

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers 3D projection)

sobol3d = Sobol(n_points=512, dimension=3, seed=0)

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(
    sobol3d.points[:, 0], sobol3d.points[:, 1], sobol3d.points[:, 2],
    s=15, color="#2E5C8A", alpha=0.7, depthshade=True,
)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_title(f"Sobol: {sobol3d.size} points in $[0,1)^3$")
plt.tight_layout()
plt.show()

## Two Functions From Computational Chemistry

Sobol's real advantage over Monte Carlo comes from the smoothness of the
integrand (via the Koksma-Hlawka inequality, which requires bounded
variation) -- not from any additive structure specifically. To see this
honestly, we compare a smooth integrand against a discontinuous one, both
grounded in computational chemistry.

**1. Product of Gaussian-type functions (smooth).** A simple,
separable Gaussian-type-orbital-like model:

$$f(\mathbf{x}) = \prod_{i=1}^d \exp(-\alpha_i x_i^2)$$

This is infinitely differentiable -- exactly the setting where Sobol's
low-discrepancy convergence rate should be most visible.

**2. A spherical interaction cutoff (discontinuous).** Interaction cutoffs
are ubiquitous in molecular simulation: many force fields and pair-potential
schemes simply ignore interactions beyond a cutoff radius $r_c$ from a
reference point, for computational efficiency. As an indicator function of
this cutoff region:

$$f(\mathbf{x}) = \mathbb{1}\left[\|\mathbf{x} - \mathbf{x}_0\|_2 < r_c\right]$$

This is discontinuous at the cutoff boundary -- not of bounded variation in
the sense required by the Koksma-Hlawka inequality -- so Sobol's theoretical
advantage is expected to largely disappear here.

In [ ]:
from scipy.special import erf, gamma

rng = np.random.default_rng(0)
DIM = 3

alpha = rng.uniform(0.5, 3.0, size=DIM)
center = np.full(DIM, 0.5)
r_cutoff = 0.3  # fully inside [0,1]^3, no boundary clipping


def gto_product(x):
    # Product of Gaussian-type functions: f(x) = prod_i exp(-alpha_i x_i^2).
    return np.exp(-(x**2 * alpha[None, :])).prod(axis=1)


def gto_product_exact():
    # Exact integral over [0,1]^d, via the error function.
    per_dim = (np.sqrt(np.pi) / (2 * np.sqrt(alpha))) * erf(np.sqrt(alpha))
    return per_dim.prod()


def cutoff_indicator(x):
    # Indicator of a spherical interaction cutoff of radius r_cutoff around center.
    return (np.linalg.norm(x - center[None, :], axis=1) < r_cutoff).astype(float)


def cutoff_indicator_exact():
    # Exact integral: volume of a d-ball of radius r_cutoff.
    return np.pi ** (DIM / 2) / gamma(DIM / 2 + 1) * r_cutoff**DIM


print(f"Exact integral of the GTO product (smooth):        {gto_product_exact():.6f}")
print(f"Exact integral of the cutoff indicator (discont.):  {cutoff_indicator_exact():.6f}")

## Comparing Against Baselines Already in `grid`

We use `Tensor1DGrids`, built from three `Trapezoidal` 1D grids (`onedgrid`),
as a structured baseline already present in the library. Like `UniformGrid`,
`Tensor1DGrids` inherits from `_HyperRectangleGrid` and is restricted to two
or three dimensions -- one more reason to run this comparison at $d=3$.
`Trapezoidal` is defined on $[-1,1]$, so its points and weights are remapped
onto $[0,1]^3$ with a simple affine transformation.

In [ ]:
from grid.onedgrid import Trapezoidal
from grid.cubic import Tensor1DGrids
from grid.basegrid import Grid

N_POWERS = [5, 7, 9, 10, 12]
N_TRIALS = 15

exact_gto = gto_product_exact()
exact_cut = cutoff_indicator_exact()

sobol_err_gto, mc_err_gto, tensor_err_gto = [], [], []
sobol_err_cut, mc_err_cut, tensor_err_cut = [], [], []
n_points_list = []

for m_pow in N_POWERS:
    N = 2 ** m_pow
    n_points_list.append(N)

    trial_sobol_gto, trial_mc_gto = [], []
    trial_sobol_cut, trial_mc_cut = [], []
    for trial in range(N_TRIALS):
        sobol = Sobol(n_points=N, dimension=DIM, seed=trial)
        trial_sobol_gto.append(abs(sobol.integrate(gto_product(sobol.points)) - exact_gto))
        trial_sobol_cut.append(abs(sobol.integrate(cutoff_indicator(sobol.points)) - exact_cut))

        mc_points = np.random.default_rng(1000 + trial).random((N, DIM))
        mc_weights = np.full(N, 1.0 / N)
        mc_grid = Grid(mc_points, mc_weights)
        trial_mc_gto.append(abs(mc_grid.integrate(gto_product(mc_points)) - exact_gto))
        trial_mc_cut.append(abs(mc_grid.integrate(cutoff_indicator(mc_points)) - exact_cut))

    sobol_err_gto.append(np.mean(trial_sobol_gto))
    mc_err_gto.append(np.mean(trial_mc_gto))
    sobol_err_cut.append(np.mean(trial_sobol_cut))
    mc_err_cut.append(np.mean(trial_mc_cut))

    m_tensor = int(round(N ** (1 / DIM)))
    oned = Trapezoidal(m_tensor)
    tensor = Tensor1DGrids(oned, oned, oned)
    tensor_points = (tensor.points + 1) / 2
    tensor_weights = tensor.weights / (2 ** DIM)
    tensor_grid = Grid(tensor_points, tensor_weights)
    tensor_err_gto.append(abs(tensor_grid.integrate(gto_product(tensor_points)) - exact_gto))
    tensor_err_cut.append(abs(tensor_grid.integrate(cutoff_indicator(tensor_points)) - exact_cut))

print("Done.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].loglog(n_points_list, mc_err_gto, "o--", color="#B0413E", label="Monte Carlo")
axes[0].loglog(n_points_list, tensor_err_gto, "^-", color="#4C9A5B", label="Tensor1DGrids (Trapezoidal)")
axes[0].loglog(n_points_list, sobol_err_gto, "s-", color="#2E5C8A", label="Sobol")
axes[0].set_xlabel("N (number of points)")
axes[0].set_ylabel("Mean absolute error")
axes[0].set_title(r"Smooth: $\prod_i \exp(-\alpha_i x_i^2)$" + f"  (d={DIM})")
axes[0].legend()
axes[0].grid(True, which="both", alpha=0.3)

axes[1].loglog(n_points_list, mc_err_cut, "o--", color="#B0413E", label="Monte Carlo")
axes[1].loglog(n_points_list, tensor_err_cut, "^-", color="#4C9A5B", label="Tensor1DGrids (Trapezoidal)")
axes[1].loglog(n_points_list, sobol_err_cut, "s-", color="#2E5C8A", label="Sobol")
axes[1].set_xlabel("N (number of points)")
axes[1].set_ylabel("Mean absolute error")
axes[1].set_title(r"Discontinuous: $\mathbb{1}[\|\mathbf{x}-\mathbf{x}_0\|<r_c]$" + f"  (d={DIM})")
axes[1].legend()
axes[1].grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.show()

On the smooth GTO product, `Sobol` outperforms both plain Monte Carlo and
the structured `Tensor1DGrids` baseline by a wide and growing margin --
exactly the setting the Koksma-Hlawka inequality predicts it should excel
in. On the discontinuous cutoff indicator, Sobol's advantage narrows
substantially and becomes less consistent from one $N$ to the next: the
bounded-variation assumption behind its theoretical guarantee no longer
holds at the cutoff boundary. Taken together, the two panels give an honest
picture: Sobol is not universally superior, but it is the clear winner for
smooth integrands, which is precisely the regime its construction targets.